In [153]:
import pandas as pd
import numpy as np

In [154]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')

In [155]:
df.shape

(7043, 21)

In [156]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [157]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

TotalCharges shouldn't be 'str'

In [158]:
pd.to_numeric(df['TotalCharges'], errors='coerce')

mask = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
).isna()

df.loc[mask, ['TotalCharges']]
df.loc[mask, 'TotalCharges'].unique()

<StringArray>
[' ']
Length: 1, dtype: str

In [159]:
(df['TotalCharges'] == ' ').sum()

np.int64(11)

In [160]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

In [161]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['TotalCharges'].isna().sum()

np.int64(0)

In [162]:
df.drop(columns='customerID', inplace=True)

### 01. Splitting

In [163]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
Y = df['Churn']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify= Y, random_state=42)

### 02. One Hot Encoding

In [164]:
ohe_cols = []

for col in X_train.columns:
    values = df[col].value_counts().size
    if values < 10:
        ohe_cols.append(col)

In [165]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
X_train_encoded = ohe.fit_transform(X_train[ohe_cols])
X_test_encoded = ohe.transform(X_test[ohe_cols])

feature_names = ohe.get_feature_names_out()

In [166]:
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=feature_names,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=feature_names,
    index=X_test.index
)

In [167]:
X_train_final = pd.concat(
    [
        X_train.drop(columns=ohe_cols),
        X_train_encoded_df
    ],
    axis=1
)
X_test_final = pd.concat(
    [
        X_test.drop(columns=ohe_cols),
        X_test_encoded_df
    ],
    axis=1
)

In [168]:
X_train_final.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 3738 to 5639
Data columns (total 46 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   tenure                                   5634 non-null   int64  
 1   MonthlyCharges                           5634 non-null   float64
 2   TotalCharges                             5634 non-null   float64
 3   gender_Female                            5634 non-null   float64
 4   gender_Male                              5634 non-null   float64
 5   SeniorCitizen_0                          5634 non-null   float64
 6   SeniorCitizen_1                          5634 non-null   float64
 7   Partner_No                               5634 non-null   float64
 8   Partner_Yes                              5634 non-null   float64
 9   Dependents_No                            5634 non-null   float64
 10  Dependents_Yes                           5634 non-null   floa

### 03. Training without Scaling

In [169]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=100)
model.fit(X_train_final, Y_train)

y_pred = model.predict(X_test_final)

print("Accuracy:", accuracy_score(Y_test, y_pred))
print("\nClassification Report:\n", classification_report(Y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(Y_test, y_pred))

Accuracy: 0.8034066713981547

Classification Report:
               precision    recall  f1-score   support

          No       0.85      0.90      0.87      1035
         Yes       0.65      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409


Confusion Matrix:
 [[927 108]
 [169 205]]


p:\machine-learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 04. Scaling

In [170]:
sc_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [181]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

scalers = [StandardScaler(), MinMaxScaler(), RobustScaler()]

for sc in scalers:

    X_train_scaled = X_train_final.copy()
    X_test_scaled = X_test_final.copy()

    X_train_scaled[sc_cols] = sc.fit_transform(
        X_train_scaled[sc_cols]
    )

    X_test_scaled[sc_cols] = sc.transform(
        X_test_scaled[sc_cols]
    )

    model = LogisticRegression(max_iter=1000)

    model.fit(X_train_scaled, Y_train)

    y_pred = model.predict(X_test_scaled)

    print(f"Scaler: {sc.__class__.__name__}")

    print("Accuracy:", accuracy_score(Y_test, y_pred), "\n")


Scaler: StandardScaler
Accuracy: 0.8005677785663591 

Scaler: MinMaxScaler
Accuracy: 0.801277501774308 

Scaler: RobustScaler
Accuracy: 0.801277501774308 

